In [25]:
from pathlib import Path

from clearml import Dataset
import pandas as pd
from tqdm.notebook import tqdm

# Params

In [34]:
freq = "15min"

# Total Power

In [2]:
data_path = Dataset.get(
    dataset_project="ForeSightNEXT/Electric Load Forecasting",
    dataset_name="household-1235",
).get_local_copy()

data_path = Path(data_path, "power.csv")

In [3]:
data = pd.read_csv(
    data_path,
    # nrows=10000,
    usecols=["_time", "_value"],
    index_col="_time",
    parse_dates=True,
    date_format="ISO8601",
)
data.rename(columns={"_value": "total"}, inplace=True)
data

,total
_time,
2020-07-28 14:01:02.030000+00:00,331
2020-07-28 14:01:04.791000+00:00,331
2020-07-28 14:01:07.733000+00:00,333
2020-07-28 14:01:10.752000+00:00,330
2020-07-28 14:01:13.731000+00:00,332
...,...
2023-01-30 13:38:05.180000+00:00,440
2023-01-30 13:38:08.224000+00:00,437
2023-01-30 13:38:11.189000+00:00,448


In [17]:
resampled_data_path = Path(f"../data/resampled-{freq}/total_power.csv")
if resampled_data_path.exists() and resampled_data_path.is_file():
    print("Resampled File already exists")
    data = pd.read_csv(
        data_path,
        index_col="_time",
        parse_dates=True,
        date_format="ISO8601",
)
else:
    resampled_data_path.parent.mkdir(exist_ok=True, parents=True)
    resampled_data = data.resample(freq).mean()
    resampled_data.to_csv(resampled_data_path)

display(resampled_data)

Resampled File already exists


,total
_time,
2020-07-28 14:01:00+00:00,331.666667
2020-07-28 14:01:10+00:00,330.750000
2020-07-28 14:01:20+00:00,330.000000
2020-07-28 14:01:30+00:00,332.000000
2020-07-28 14:01:40+00:00,330.750000
...,...
2023-01-30 13:37:30+00:00,435.666667
2023-01-30 13:37:40+00:00,440.666667
2023-01-30 13:37:50+00:00,436.750000


# Devices

In [19]:
shiftable = {
    "DECT210 Waschmaschine": "uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb",    
    "DECT200 Waschmaschine": "uri:urn:4b29c04c920141e8",
    "Smart Switch 6 Spülmaschine": "uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad",    
    "DECT200 Spülmaschine": "uri:urn:cc256ae649904024",
    # "Smart Switch 6 Thermomix": "uri:urn:5b59894b-805d-4d7c-96c9-8ab42a18c3c7",
}

hh_root = Path("../../../datasets/ForeSight/household-1235/")
ts_dirs = {k: Path(hh_root, fn) for k, fn in shiftable.items()}
ts_dirs

{'DECT210 Waschmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb'),
 'DECT200 Waschmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8'),
 'Smart Switch 6 Spülmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad'),
 'DECT200 Spülmaschine': PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024')}

In [20]:
for k, td in ts_dirs.items():
    print(k)
    display(list(td.iterdir()))
    print("\n")

DECT210 Waschmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb/socket-on.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb/sensor_160-value.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:6a112240-117e-4ec3-a129-5bc90908aedb/sensor_163-value.csv')]



DECT200 Waschmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/temperature.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/onOff.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/energy.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:4b29c04c920141e8/power.csv')]



Smart Switch 6 Spülmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad/sensor_582-value.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:e91f9319-71af-4ddb-ab7d-fb47b45d69ad/socket-on.csv')]



DECT200 Spülmaschine


[PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/temperature.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/onOff.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/energy.csv'),
 PosixPath('../../../datasets/ForeSight/household-1235/uri:urn:cc256ae649904024/power.csv')]

In [30]:
def read_tables(thing_name: str, freq: str, **kwargs):
    frames = []
    for p in tqdm(
        list(ts_dirs[thing_name].iterdir())# + 
    ):
        if p.suffix == ".csv" and any([
            p.stem == "energy",
            p.stem == "power",
            p.stem.startswith("sensor"),
        ]):
            df = pd.read_csv(
                p,
                usecols=["_time", "_value"],
                index_col="_time",
                parse_dates=True,
                date_format="ISO8601",
                **kwargs,
            )
            df.columns = [p.stem]
            df.astype(float)
            df = df.resample(freq).mean()
            frames.append(df)
    return pd.concat(frames, axis=1)


def cached_resample(thing_name: str, freq=str):
    out_path = Path(f"../data/resampled-{freq}/{thing_name}.csv")
    if out_path.exists() and out_path.is_file():
        print("reading from cache...")
        time_series = pd.read_csv(
            out_path, 
            index_col="_time",
            parse_dates=True,
            date_format="ISO8601"
        )
    else:
        print("resampling...")
        time_series = read_tables(thing_name=thing_name, freq=freq)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        time_series.to_csv(out_path)

    return time_series

In [33]:
series = {thing_name: cached_resample(thing_name, freq) for thing_name in ts_dirs.keys()}

resampling...


  0%|          | 0/3 [00:00<?, ?it/s]

resampling...


  0%|          | 0/4 [00:00<?, ?it/s]

resampling...


  0%|          | 0/2 [00:00<?, ?it/s]

resampling...


  0%|          | 0/4 [00:00<?, ?it/s]